# Python for physics: exercise sheet and reference

[Workshop index](../README.md) | [OS setup guides](../README.md)

This notebook is the single Python teaching reference. Start with the short examples, then continue to paths, fitting, uncertainties, and the final exercises.

## Before running

1. Create the project `.venv` and install [requirements.txt](../requirements.txt) using your OS guide.
2. Follow the interpreter/kernel checklist in the [OS setup guide](../README.md).
3. Inspect source before running a downloaded notebook. Code cells execute; Markdown cells explain.
4. Run top to bottom. Restart the kernel and run all before sharing: old output can hide missing definitions.

| Package | Role |
| --- | --- |
| [NumPy](https://numpy.org/doc/stable/user/quickstart.html) | Arrays and numerical operations |
| [Matplotlib](https://matplotlib.org/stable/tutorials/pyplot.html) | Plots and exported figures |
| [Pandas](https://pandas.pydata.org/docs/user_guide/10min.html) | Labelled tables and CSV data |
| [SciPy](https://docs.scipy.org/doc/scipy/) | Fitting, optimisation, and signal analysis |
| Astropy | Astronomy tools and units; not required for these exercises |
| ipykernel | Notebook execution in the selected environment |

The requirements are unpinned: they describe a starting environment, not an exact version lockfile. See the [Jupyter documentation](https://docs.jupyter.org/en/latest/) for notebook workflow.

**Files written:** `data_backup.csv` and `fit_result.pdf` in the working directory. Existing files with these names will be overwritten. These are generated practice output, not independent backups. Keep raw measurements separate.

### Libraries

Run this cell first. These libraries will be used in the examples below.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import find_peaks

### A simple function

A function is a named piece of code. It can take an input and return an answer. Here, **x** is the input, and the function returns the sine of **x**.

In [ ]:
def function(x):
    return np.sin(x)

### Plotting a function

Make a set of x values, use the function to calculate y values, then plot them. Try changing the range or number of points. See the [Matplotlib pyplot guide](https://matplotlib.org/stable/tutorials/pyplot.html).

In [ ]:
x = np.linspace(0, 10, 100)
y = function(x)

plt.plot(x, y)
plt.xlabel("x (radians)")
plt.ylabel("sin(x)")
plt.title("A simple function")
plt.grid(True)
plt.show()

### NumPy arrays

NumPy arrays store numbers. You can calculate with all the values at once and use built-in functions such as **mean** and **max**. See the [NumPy quickstart](https://numpy.org/doc/stable/user/quickstart.html).

In [ ]:
measurements = np.array([1.2, 1.4, 1.3, 1.5])
print("Measurements:", measurements)
print("Mean:", np.mean(measurements))
print("Maximum:", np.max(measurements))

### A small Pandas table

Pandas stores data in a table called a DataFrame. Columns have names, which makes it easier to keep measurements organised. See the [Pandas ten-minute guide](https://pandas.pydata.org/docs/user_guide/10min.html).

In [ ]:
data = pd.DataFrame({
    "time_s": [0, 1, 2, 3],
    "position_m": [0.0, 0.8, 2.1, 2.9],
})
display(data)

# Select one column by its name.
print("Mean position:", data["position_m"].mean(), "m")

### Finding peaks with SciPy

**find_peaks** finds values that are larger than the nearby values. The result gives the positions of the peaks in the array. See the [SciPy find_peaks reference](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.find_peaks.html).

In [ ]:
signal = np.array([0, 1, 0, 0, 3, 0, 1, 0])
peak_positions, properties = find_peaks(signal)

print("Peak positions:", peak_positions)
print("Peak values:", signal[peak_positions])

### Plotting the peaks

The peak positions can be marked on a plot. This makes it easy to check what the function found.

In [ ]:
sample_number = np.arange(len(signal))

plt.plot(sample_number, signal, marker="o")
plt.plot(peak_positions, signal[peak_positions], "rx", label="peaks")
plt.xlabel("Sample number")
plt.ylabel("Signal")
plt.legend()
plt.grid(True)
plt.show()

### Saving and reading data

A function can also perform a task such as saving data. The first function saves two columns; the second reads those columns back. This creates or overwrites `data_backup.csv` in the working directory; despite its name, it is not an independent backup.

In [ ]:
def save_data(time, y_values, filename):
    """Save time and y values as two columns in a CSV file."""
    data_to_save = np.column_stack((time, y_values))
    np.savetxt(filename, data_to_save, delimiter=",", header="time,y_values", comments="")
    print(f"Data saved to {filename}")

def read_data(filename):
    """Read time and y values from a two-column CSV file."""
    data_from_file = np.loadtxt(filename, delimiter=",", skiprows=1)
    time = data_from_file[:, 0]
    y_values = data_from_file[:, 1]
    return time, y_values

# Save the x and y values created earlier, then read them back.
save_data(x, y, "data_backup.csv")
loaded_x, loaded_y = read_data("data_backup.csv")
print("First three y values:", loaded_y[:3])

### Try it yourself

1. Change the function to return **np.cos(x)**.
2. Add more measurements to the NumPy array and calculate their mean.
3. Add another column to the Pandas table.
4. Change the values in **signal** and see which peaks are found.
5. Change the plot labels to describe a measurement from one of your labs.

## Working directories and relative paths

For the fitting exercise, use `Example_Code` as the working directory. In a terminal, activate the workshop environment from `Computer_Literacy`, then `cd Example_Code`. In a notebook, check the directory in the cell below. If needed, use `%cd` with the path to your own `Example_Code` folder in a separate cell; do not hard-code someone else's home directory.

Use paths relative to the notebook kernel's working directory; Python documents filesystem paths in [`pathlib`](https://docs.python.org/3/library/pathlib.html). Use the supplied [synthetic motion dataset](linear_motion.csv) for the fitting example.

**Nested-path exercise:** create a folder named `Data_folder` and copy [linear_motion.csv](linear_motion.csv) into it. From `Example_Code`, load it as `Data_folder/linear_motion.csv`; from inside that folder, use `linear_motion.csv`. Explain why the relative path changes. Preserve the original data and document transformations.

In [ ]:
from pathlib import Path

print(Path.cwd())

## Fitting a physical model

The supplied CSV is **synthetic teaching data, not a real experiment**. Six deterministic positions were chosen near $x=0.4+1.2t$ with small illustrative deviations. There is no random seed or instrument calibration. Never present them as laboratory measurements.

| Column | Meaning |
| --- | --- |
| `time_s` | Time in seconds, treated as exact |
| `position_m` | Simulated position in metres |
| `sigma_position_m` | Assumed independent one-standard-deviation position uncertainty in metres |

Fit $x(t)=vt+x_0$. The next cell reads `linear_motion.csv` and creates or overwrites `fit_result.pdf` in the working directory. The imports are repeated here so this worked example can also be run independently of the earlier cells. The fitting API and uncertainty parameters are documented in [`scipy.optimize.curve_fit`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

data_path = Path("linear_motion.csv")
if not data_path.is_file():
    raise FileNotFoundError(
        "Open Example_Code as the working directory to load linear_motion.csv."
    )

motion_data = pd.read_csv(data_path)
time_s = motion_data["time_s"].to_numpy()
position_m = motion_data["position_m"].to_numpy()
sigma_position_m = motion_data["sigma_position_m"].to_numpy()

def position(time, velocity, intercept):
    return velocity * time + intercept

parameters, covariance = curve_fit(
    position, time_s, position_m, sigma=sigma_position_m, absolute_sigma=True
)
velocity, intercept = parameters
uncertainties = np.sqrt(np.diag(covariance))
residuals = position_m - position(time_s, *parameters)
dof = len(time_s) - len(parameters)
chi_squared = np.sum((residuals / sigma_position_m) ** 2)

print(f"velocity = {velocity:.3f} +/- {uncertainties[0]:.3f} m/s")
print(f"intercept = {intercept:.3f} +/- {uncertainties[1]:.3f} m")
print(f"chi-squared / dof = {chi_squared:.2f} / {dof}")

fig, axes = plt.subplots(2, 1, sharex=True, constrained_layout=True)
axes[0].errorbar(time_s, position_m, yerr=sigma_position_m, fmt="o", label="synthetic data")
axes[0].plot(time_s, position(time_s, *parameters), label="linear fit")
axes[0].set_ylabel("Position (m)")
axes[0].legend()
axes[1].errorbar(time_s, residuals, yerr=sigma_position_m, fmt="o")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Residual (m)")
fig.savefig("fit_result.pdf")
plt.show()


### Expected output and interpretation

```text
velocity = 1.198 +/- 0.024 m/s
intercept = 0.413 +/- 0.072 m
chi-squared / dof = 1.80 / 4
```

The upper plot shows six observations and a fitted line. The lower plot shows observed minus predicted position; a positive residual lies above the model. Use the [LaTeX guide](../LaTeX/LaTeX.pdf) to include `fit_result.pdf` in a report.

Covariance-based uncertainties assume the model, independent Gaussian errors, supplied standard deviations, and negligible time uncertainty. `absolute_sigma=True` treats the supplied uncertainties as absolute rather than rescaling them from residuals.

$$\chi^2=\sum_i\left(\frac{x_i-x(t_i)}{\sigma_i}\right)^2,\qquad \nu=N-p=6-2=4.$$

Small residuals or low chi-squared do not prove a model is correct. This tidy synthetic dataset is not statistically representative. Structured residuals can reveal a wrong model; outliers, correlated errors, and uncertainty in both axes need more careful methods.

### Uncertainty vocabulary

Standard deviation describes scatter. Under suitable independent sampling, the standard error of a mean is $s/\sqrt{N}$; it does not remove systematic effects. For independent variables and small uncertainties,

$$\sigma_f^2\simeq\sum_i\left(\frac{\partial f}{\partial q_i}\right)^2\sigma_{q_i}^2.$$

Correlations add covariance terms. State assumptions, units, and sensible significant figures.

## Further exercises and success checks

1. Convert peak indices to physical times using sample spacing before labelling an axis in seconds.
2. Complete the nested-folder loading exercise above.
3. In a copy of the fit, double every position uncertainty. The best-fit line should stay unchanged, parameter uncertainties double, and chi-squared becomes one quarter.
4. Add a curved trend to a copy of the positions. Inspect residuals and explain why a plausible straight line is not enough.
5. Export the figure and use the [LaTeX guide](../LaTeX/LaTeX.pdf) to include it in a report; describe the synthetic origin, units, error bars, and assumptions.
6. Restart and run all cells; ask a partner to reproduce your project without your help.

## Troubleshooting and reproducibility

| Symptom | Check / action |
| --- | --- |
| `ModuleNotFoundError` | Follow the interpreter/kernel checks in the editor guide |
| `FileNotFoundError` | Print `Path.cwd()`; check filename case and relative path |
| `NameError` | Restart and run cells in order; a definition may be missing |
| CSV has no numbers | Check delimiter, header, and whether the file is empty |
| Plot missing | Check editor/backend output and the exported PDF |
| Results change after restart | Check stale state, out-of-order cells, and unrecorded randomness |

Record Python version, OS, package versions, data provenance, and run instructions. `python -m pip freeze` records the active environment; save it separately as `requirements-tested.txt`, not over the workshop requirements. Inspect it for private/local paths before sharing. Do not commit `.venv` or private data; keep independent backups.